### Import Libraries

In [1]:
# ==========================================
# UK BUSINESS PORTAL SCRAPER
# ==========================================

# Selenium
from selenium import webdriver
from selenium.webdriver.common.by import By

# HTML Parsing
from bs4 import BeautifulSoup

# Data Manipulation
import pandas as pd

# Utilities
import time
from datetime import datetime

# Connection to the database
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

### Launch Chrome Driver

In [2]:
# ==========================================
# START BROWSER SESSION
# ==========================================

driver = webdriver.Chrome()

driver.maximize_window()

print("Chrome launched successfully")

Chrome launched successfully


### Navigate To Homepage

In [3]:
# ==========================================
# LOAD HOMEPAGE
# ==========================================

homepage = "https://ukbusinessportal.co.uk/"

driver.get(homepage)

time.sleep(5)

print("Homepage loaded")

Homepage loaded


### Extract Category URLs

In [4]:
# ==========================================
# CREATE SOUP OBJECT FOR HTML PARSING
# ==========================================

soup = BeautifulSoup(
    driver.page_source,
    "html.parser"
)

### Find Category Links

In [5]:
# ==========================================
# INSPECT CATEGORY LINKS
# ==========================================

links = soup.find_all("a", href=True)

print(f"Total links found: {len(links)}")

Total links found: 141


### Extract Category URLs

In [6]:
# ==========================================
# FIND CATEGORY URLS
# ==========================================

for link in links:

    href = link["href"]

    if "/category/" in href:

        print(href)

https://ukbusinessportal.co.uk/category/accountants/
https://ukbusinessportal.co.uk/category/advertising-marketing/
https://ukbusinessportal.co.uk/category/architects/
https://ukbusinessportal.co.uk/category/asbestos-surveyor/
https://ukbusinessportal.co.uk/category/bathrooms/
https://ukbusinessportal.co.uk/category/beauty/
https://ukbusinessportal.co.uk/category/blinds-shutters/
https://ukbusinessportal.co.uk/category/boiler-maintenance/
https://ukbusinessportal.co.uk/category/builders/
https://ukbusinessportal.co.uk/category/business-coaching/
https://ukbusinessportal.co.uk/category/car-repair/
https://ukbusinessportal.co.uk/category/car-sales/
https://ukbusinessportal.co.uk/category/carpenters/
https://ukbusinessportal.co.uk/category/catering/
https://ukbusinessportal.co.uk/category/charity/
https://ukbusinessportal.co.uk/category/chiropractors/
https://ukbusinessportal.co.uk/category/cleaning/
https://ukbusinessportal.co.uk/category/computer-repair/
https://ukbusinessportal.co.uk/c

### Build Categories DataFrame

In [7]:
# ==========================================
# CREATE CATEGORY DATAFRAME
# ==========================================

category_data = []

for link in links:

    href = link.get("href")

    if href and "/category/" in href:

        category_name = (
            link.get_text(strip=True)
        )

        category_data.append({
            "category_name": category_name,
            "category_url": href
        })

categories_df = (
    pd.DataFrame(category_data)
    .drop_duplicates()
    .reset_index(drop=True)
)

categories_df.head()

,category_name,category_url
0,Accountants,https://ukbusinessportal.co.uk/category/accoun...
1,Advertising & Marketing,https://ukbusinessportal.co.uk/category/advert...
2,Architects,https://ukbusinessportal.co.uk/category/archit...
3,Asbestos Surveyor,https://ukbusinessportal.co.uk/category/asbest...
4,Bathrooms,https://ukbusinessportal.co.uk/category/bathro...


### Verify Categories

In [8]:
# ==========================================
# CATEGORY SUMMARY
# ==========================================

print(
    f"Categories discovered: {len(categories_df)}"
)

categories_df

Categories discovered: 64


,category_name,category_url
0,Accountants,https://ukbusinessportal.co.uk/category/accoun...
1,Advertising & Marketing,https://ukbusinessportal.co.uk/category/advert...
2,Architects,https://ukbusinessportal.co.uk/category/archit...
3,Asbestos Surveyor,https://ukbusinessportal.co.uk/category/asbest...
4,Bathrooms,https://ukbusinessportal.co.uk/category/bathro...
...,...,...
59,Surveyors,https://ukbusinessportal.co.uk/category/survey...
60,Taxis,https://ukbusinessportal.co.uk/category/taxis/
61,Telecoms,https://ukbusinessportal.co.uk/category/telecoms/
62,Tree Surgeon,https://ukbusinessportal.co.uk/category/tree-s...


### Business Extraction Function

In [9]:
# ==========================================
# BUSINESS EXTRACTION FUNCTION
# ==========================================

def extract_business_card(card, category_name):

    name = None
    website = None
    phone = None
    email = None
    logo_url = None
    profile_url = None

    # Business Name
    h3 = card.find("h3")

    if h3:
        name = h3.get_text(strip=True)

    # Logo URL
    img = card.find("img")

    if img:
        logo_url = img.get("src")

    # Links
    for a in card.find_all("a", href=True):

        href = a["href"]

        if "/business/" in href:
            profile_url = href

        elif href.startswith("http"):
            website = href

        elif href.startswith("tel:"):
            phone = href.replace("tel:", "")

        elif href.startswith("mailto:"):
            email = href.replace("mailto:", "")

    return {
        "category": category_name,
        "business_name": name,
        "website": website,
        "phone": phone,
        "email": email,
        "logo_url": logo_url,
        "profile_url": profile_url
    }

### Category Scraper Function

In [10]:
# ==========================================
# CATEGORY SCRAPER
# ==========================================

def scrape_category(category_name, category_url):

    driver.get(category_url)

    time.sleep(3)

    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    cards = soup.select("div.shadow-custom")

    businesses = []

    for card in cards:

        if card.find("h3") is None:
            continue

        businesses.append(
            extract_business_card(
                card,
                category_name
            )
        )

    return pd.DataFrame(businesses)

### Scrape All Categories

In [11]:
# ==========================================
# SCRAPE ALL CATEGORIES
# ==========================================

all_dfs = []

scrape_log = []

for _, row in categories_df.iterrows():

    category_name = row["category_name"]
    category_url = row["category_url"]

    print(
        f"Scraping {category_name}"
    )

    temp_df = scrape_category(
        category_name,
        category_url
    )

    all_dfs.append(temp_df)

    scrape_log.append({
        "category": category_name,
        "records_scraped": len(temp_df)
    })

Scraping Accountants
Scraping Advertising & Marketing
Scraping Architects
Scraping Asbestos Surveyor
Scraping Bathrooms
Scraping Beauty
Scraping Blinds & Shutters
Scraping Boiler Maintenance
Scraping Builders
Scraping Business Coaching
Scraping Car Repair
Scraping Car Sales
Scraping Carpenters
Scraping Catering
Scraping Charity
Scraping Chiropractors
Scraping Cleaning
Scraping Computer Repair
Scraping Construction
Scraping Consultancy
Scraping Counselling
Scraping Dentists
Scraping Driveways & Paving Contractors
Scraping Driving Instructors
Scraping Education
Scraping Electricians
Scraping Entertainment
Scraping Estate & Letting Agents
Scraping Financial Adviser
Scraping Florists
Scraping Furniture
Scraping Garage Doors
Scraping Gardeners
Scraping Hairdressers
Scraping Health & Wellbeing
Scraping Hypnotherapy
Scraping Interior Design
Scraping Kitchens
Scraping Life Coaching
Scraping Locksmith
Scraping Luxury Interiors
Scraping Manufacturing
Scraping Mortgage Broker
Scraping Osteopathy


### Create Master Dataset

In [12]:
# ==========================================
# COMBINE ALL DATA
# ==========================================

master_df = pd.concat(
    all_dfs,
    ignore_index=True
)

master_df.head()

,category,business_name,website,phone,email,logo_url,profile_url
0,Accountants,Wellden Turnbull,https://www.wtca.co.uk/,01932 868444,info@wtca.co.uk,https://ukbusinessportal.co.uk/wp-content/uplo...,https://ukbusinessportal.co.uk/business/wellde...
1,Accountants,Elsby & Co,https://www.elsbyandco.co.uk/,01933 312950,help@elsbyandco.co.uk,https://ukbusinessportal.co.uk/wp-content/uplo...,https://ukbusinessportal.co.uk/business/elsby-co/
2,Accountants,Hodson & Co,https://www.hodsonandco.co.uk/,01903 238628,info@hodsonandco.co.uk,https://ukbusinessportal.co.uk/wp-content/uplo...,https://ukbusinessportal.co.uk/business/hodson...
3,Accountants,Rawlinson Pryde & Partners,https://rppaccounts.co.uk/,01234 300500,mail@rppaccounts.co.uk,https://ukbusinessportal.co.uk/wp-content/uplo...,https://ukbusinessportal.co.uk/business/rawlin...
4,Accountants,Robson Laidler Accountants,https://www.robson-laidler.co.uk/,0191 281 8191,ba@robson-laidler.co.uk,https://ukbusinessportal.co.uk/wp-content/uplo...,https://ukbusinessportal.co.uk/business/robson...


### Add Audit Columns

In [13]:
# ==========================================
# ADD METADATA
# ==========================================

master_df["scrape_timestamp"] = (
    datetime.now()
    .strftime("%Y-%m-%d %H:%M:%S")
)

master_df["scrape_date"] = (
    datetime.now()
    .strftime("%Y-%m-%d")
)

### Data Quality Checks

In [14]:
# ==========================================
# DATA QUALITY CHECKS
# ==========================================

print("\nDataset Shape")
print(master_df.shape)

print("\nMissing Values")
print(master_df.isna().sum())

print("\nDuplicate Records")
print(master_df.duplicated().sum())


Dataset Shape
(307, 9)

Missing Values
category            0
business_name       0
website             1
phone               4
email               0
logo_url            0
profile_url         0
scrape_timestamp    0
scrape_date         0
dtype: int64

Duplicate Records
0


### Scrape Summary

In [15]:
# ==========================================
# SCRAPE SUMMARY
# ==========================================

scrape_log_df = pd.DataFrame(
    scrape_log
)

scrape_log_df

,category,records_scraped
0,Accountants,6
1,Advertising & Marketing,6
2,Architects,4
3,Asbestos Surveyor,2
4,Bathrooms,5
...,...,...
59,Surveyors,12
60,Taxis,9
61,Telecoms,1
62,Tree Surgeon,3


### Data Quality Report

In [16]:
# ==========================================
# DATA QUALITY REPORT
# ==========================================

dq_report = pd.DataFrame({
    "column": master_df.columns,
    "missing_values": master_df.isna().sum().values,
    "missing_pct": (
        master_df.isna().mean() * 100
    ).round(2).values
})

dq_report

,column,missing_values,missing_pct
0,category,0,0.00
1,business_name,0,0.00
2,website,1,0.33
3,phone,4,1.30
4,email,0,0.00
5,logo_url,0,0.00
6,profile_url,0,0.00
7,scrape_timestamp,0,0.00
8,scrape_date,0,0.00


### Save CSV

In [17]:
# ==========================================
# EXPORT DATASET
# ==========================================

master_df.to_csv(
    "uk_business_portal.csv",
    index=False
)

print(
    f"Exported {len(master_df)} records"
)

Exported 307 records


### Close Browser

In [18]:
# ==========================================
# CLEANUP
# ==========================================

driver.quit()

print("Browser closed")

Browser closed


### Loading the scraped data into the database

In [19]:
load_dotenv()
try:
    db_username = os.getenv("db_username")
    db_password = os.getenv("db_password")
    db_host = os.getenv("db_host")
    db_port = os.getenv("db_port")
    db_name = os.getenv("db_name")

    url_connection = f"postgresql://{db_username}:{db_password}@{db_host}:{db_port}/{db_name}"
    engine = create_engine(url_connection)
    table_name = "businesses"

    # write dataframe to postgres
    master_df.to_sql(table_name,engine,if_exists = 'replace', index = False)
    print(f"Data successfully loaded into the {db_name} database")
except Exception as e:
    print(e)    

Data successfully loaded into the uk_business database
